# samples

> Small DB-shaped sample blocks for offline architecture development.

The goal is to develop the package, DB loaders, analytics, and visualizations against a tiny representative dataset before committing to expensive live block ingestion. A sample block is intentionally plain data: a Bitcoin-Core-ish `block` dict plus `transactions` rows shaped like `block_fee_rows()` and `db.insert_transaction_rows()`.

In [ ]:
#| default_exp samples

In [ ]:
#| hide
from nbdev.showdoc import *

## Contract

Every sample returns the same shape:

```python
{
    "block": {"hash": "...", "height": 842913, "time": 1747733650},
    "transactions": [block_fee_rows_style_dict, ...],
}
```

That pair can go directly into `insert_block()` and `insert_transaction_rows()`, while visualizations can aggregate the same rows without Postgres.

In [ ]:
#| eval: false
from slashbtc.samples import sample_block, sample_to_db_rows

sample = sample_block("synthetic_modern_busy")
block, rows = sample_to_db_rows(sample)

## Coverage

Synthetic samples are the default mini-chain for architecture work. Real specimens are packaged separately to keep us honest when we want a mainnet check. `synthetic_empty` and `synthetic_zero_fee_era` protect edge cases, `synthetic_modern_busy` and `synthetic_congestion_tail` drive the fee-market UI, and packaged mainnet fixtures prove the shape can come from Bitcoin Core verbosity `3`.

## Loading Postgres

The same sample set can seed the current warehouse schema. This keeps DB work, transforms, and charts attached to one base dataset.

In [ ]:
#| eval: false
from slashbtc.samples import load_samples_into_db

block_keys = load_samples_into_db(conn)

## Refreshing Fixtures

Use live RPC only when minting a new compact sample from a real block.

In [ ]:
#| eval: false
# Reads BITCOIN_RPC_URL and BITCOIN_RPC_COOKIE from .env by default.
# slashbtc-capture-sample-block 947663 \
#   --name mainnet_recent_947663 \
#   --out slashbtc/data/sample_blocks/mainnet_recent_947663.json.gz

In [ ]:
#| export
import argparse
import gzip
import hashlib
import json
import math
import os
import random
from decimal import Decimal
from importlib import resources
from pathlib import Path
from typing import Any, Iterable, Mapping

from dotenv import load_dotenv

from slashbtc.core import Chain
from slashbtc.transform import block_fee_rows

SAMPLE_PACKAGE = "slashbtc"
SAMPLE_DIR = "data/sample_blocks"
DEFAULT_SAMPLE_NAMES = (
    "synthetic_empty",
    "synthetic_zero_fee_era",
    "synthetic_calm",
    "synthetic_modern_busy",
    "synthetic_congestion_tail",
)

SYNTHETIC_PROFILES: dict[str, dict[str, Any]] = {
    "synthetic_modern_busy": {
        "height": 842913,
        "time": 1747733650,
        "tx_count": 2847,
        "median_fee_sat_vb": 18,
        "sigma": 0.62,
        "seed": 842913,
        "tail_share": 0.05,
        "tail_multiplier": 7.5,
        "zero_fee_share": 0.0,
        "consolidation_share": 0.08,
    },
    "synthetic_calm": {
        "height": 700001,
        "time": 1631333672,
        "tx_count": 1850,
        "median_fee_sat_vb": 3,
        "sigma": 0.48,
        "seed": 700001,
        "tail_share": 0.03,
        "tail_multiplier": 4.0,
        "zero_fee_share": 0.01,
        "consolidation_share": 0.08,
    },
    "synthetic_congestion_tail": {
        "height": 800008,
        "time": 1690751730,
        "tx_count": 3600,
        "median_fee_sat_vb": 72,
        "sigma": 0.85,
        "seed": 800008,
        "tail_share": 0.10,
        "tail_multiplier": 9.0,
        "zero_fee_share": 0.0,
        "consolidation_share": 0.08,
    },
    "synthetic_zero_fee_era": {
        "height": 100000,
        "time": 1293623863,
        "tx_count": 3,
        "median_fee_sat_vb": 0,
        "sigma": 0.0,
        "seed": 100000,
        "tail_share": 0.0,
        "tail_multiplier": 1.0,
        "zero_fee_share": 1.0,
        "consolidation_share": 0.0,
    },
    "synthetic_empty": {
        "height": 10000,
        "time": 1238988213,
        "tx_count": 0,
        "median_fee_sat_vb": 0,
        "sigma": 0.0,
        "seed": 10000,
        "tail_share": 0.0,
        "tail_multiplier": 1.0,
        "zero_fee_share": 1.0,
        "consolidation_share": 0.0,
    },
}


def sample_block_names() -> list[str]:
    """List available synthetic and packaged sample block names."""
    names = set(SYNTHETIC_PROFILES)
    try:
        fixture_root = resources.files(SAMPLE_PACKAGE).joinpath(SAMPLE_DIR)
        for item in fixture_root.iterdir():
            if item.name.endswith(".json.gz"):
                names.add(item.name.removesuffix(".json.gz"))
            elif item.name.endswith(".json"):
                names.add(item.name.removesuffix(".json"))
    except FileNotFoundError:
        pass
    return sorted(names)


def sample_block(name: str = "synthetic_modern_busy") -> dict[str, Any]:
    """Return one sample block with DB-compatible `block` and `transactions` rows."""
    if name in SYNTHETIC_PROFILES:
        return synthetic_sample_block(name)
    return _packaged_sample_block(name)


def sample_blocks(names: Iterable[str] | None = None) -> list[dict[str, Any]]:
    """Return a small representative chain slice for notebooks and integration tests."""
    return [sample_block(name) for name in (tuple(names) if names is not None else DEFAULT_SAMPLE_NAMES)]


def sample_to_db_rows(sample: Mapping[str, Any]) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    """Return the `(block, transaction_rows)` pair expected by `db.insert_*` helpers."""
    return dict(sample["block"]), [dict(row) for row in sample["transactions"]]


def synthetic_sample_block(name: str = "synthetic_modern_busy") -> dict[str, Any]:
    """Generate a deterministic DB-shaped sample block by profile name."""
    try:
        profile = SYNTHETIC_PROFILES[name]
    except KeyError as exc:
        available = ", ".join(sorted(SYNTHETIC_PROFILES))
        raise KeyError(f"unknown synthetic sample {name!r}; available: {available}") from exc

    rng = random.Random(profile["seed"])
    non_coinbase_rows = _synthetic_transaction_rows(name, profile, rng)
    total_fees_sats = sum(row["fee_sats"] or 0 for row in non_coinbase_rows)
    coinbase = _coinbase_row(name, profile, total_fees_sats)
    rows = [coinbase, *non_coinbase_rows]
    for i, row in enumerate(rows):
        row["position_in_block"] = i

    return {
        "schema": "slashbtc.sample_block.v1",
        "name": name,
        "source": "synthetic",
        "block": _sample_block_header(name, profile),
        "transactions": rows,
    }


def load_sample_block(path: str | Path) -> dict[str, Any]:
    """Load a packaged or local sample block from `.json` or `.json.gz`."""
    path = Path(path)
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        return json.load(f)


def save_sample_block(sample: Mapping[str, Any], path: str | Path) -> Path:
    """Save a sample block with deterministic JSON formatting."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "wt", encoding="utf-8") as f:
        json.dump(sample, f, sort_keys=True, separators=(",", ":"))
        f.write("\n")
    return path


def sample_block_from_rpc(block: Mapping[str, Any], name: str | None = None, source: str = "bitcoin-core") -> dict[str, Any]:
    """Convert a Bitcoin Core verbosity=3 block into the canonical sample-block schema."""
    height = block.get("height")
    sample_name = name or f"mainnet_{height}"
    return {
        "schema": "slashbtc.sample_block.v1",
        "name": sample_name,
        "source": source,
        "block": {
            "hash": block["hash"],
            "height": height,
            "time": block["time"],
            "tx_count": len(block.get("tx", [])),
        },
        "transactions": block_fee_rows(dict(block)),
    }


def capture_sample_block(
    chain: Chain,
    height_or_hash: int | str,
    path: str | Path | None = None,
    name: str | None = None,
) -> dict[str, Any]:
    """Fetch one real block once, optionally saving it as a reusable sample block."""
    sample = sample_block_from_rpc(chain.block(height_or_hash, verbosity=3), name=name)
    if path is not None:
        save_sample_block(sample, path)
    return sample


def load_samples_into_db(conn, names: Iterable[str] | None = None) -> list[int]:
    """Create the current schema and insert sample blocks into Postgres."""
    from slashbtc.db import create_schema, insert_block, insert_transaction_rows

    create_schema(conn)
    block_keys = []
    for sample in sample_blocks(names):
        block, rows = sample_to_db_rows(sample)
        block_key = insert_block(conn, block)
        insert_transaction_rows(conn, block_key, rows)
        block_keys.append(block_key)
    return block_keys


def capture_sample_block_main(argv: list[str] | None = None) -> int:
    """CLI entrypoint for minting compact DB-shaped sample blocks from Bitcoin Core."""
    load_dotenv()
    parser = argparse.ArgumentParser(description="Capture a DB-shaped sample block from Bitcoin Core RPC.")
    parser.add_argument("height_or_hash", help="Block height or block hash to capture.")
    parser.add_argument("--out", required=True, help="Destination .json or .json.gz sample path.")
    parser.add_argument("--name", help="Sample name stored in the fixture.")
    parser.add_argument("--url", default=os.getenv("BITCOIN_RPC_URL"), help="Bitcoin Core RPC URL.")
    parser.add_argument("--cookie", default=os.getenv("BITCOIN_RPC_COOKIE"), help="Cookie path or raw user:password.")
    parser.add_argument("--rpc-user", default=os.getenv("BITCOIN_RPC_USER"), help="RPC username.")
    parser.add_argument("--rpc-password", default=os.getenv("BITCOIN_RPC_PASSWORD"), help="RPC password.")
    parser.add_argument("--timeout", type=float, default=120.0, help="RPC timeout in seconds.")
    args = parser.parse_args(argv)

    if not args.url:
        parser.error("--url or BITCOIN_RPC_URL is required")

    auth = _rpc_auth(args.cookie, args.rpc_user, args.rpc_password)
    height_or_hash = int(args.height_or_hash) if args.height_or_hash.isdigit() else args.height_or_hash
    with Chain(url=args.url, auth=auth, timeout=args.timeout) as chain:
        sample = capture_sample_block(chain, height_or_hash, path=args.out, name=args.name)
    print(
        f"saved {args.out} "
        f"height={sample['block']['height']} "
        f"txs={len(sample['transactions'])}"
    )
    return 0


def _packaged_sample_block(name: str) -> dict[str, Any]:
    root = resources.files(SAMPLE_PACKAGE).joinpath(SAMPLE_DIR)
    candidates = [root.joinpath(f"{name}.json.gz"), root.joinpath(f"{name}.json")]
    for candidate in candidates:
        if candidate.is_file():
            with resources.as_file(candidate) as path:
                return load_sample_block(path)
    available = ", ".join(sample_block_names())
    raise KeyError(f"unknown sample block {name!r}; available: {available}")


def _sample_block_header(name: str, profile: Mapping[str, Any]) -> dict[str, Any]:
    return {
        "hash": _fake_hash(f"block:{name}:{profile['height']}"),
        "height": profile["height"],
        "time": profile["time"],
        "tx_count": profile["tx_count"] + 1,
    }


def _synthetic_transaction_rows(name: str, profile: Mapping[str, Any], rng: random.Random) -> list[dict[str, Any]]:
    rows = []
    median = max(float(profile["median_fee_sat_vb"]), 0.0)
    log_median = math.log(max(median, 0.1))
    for i in range(1, profile["tx_count"] + 1):
        vsize = _synthetic_vsize(profile, rng)
        if rng.random() < profile["zero_fee_share"] or median == 0:
            fee_rate = 0.0
        else:
            fee_rate = rng.lognormvariate(log_median, profile["sigma"])
            if rng.random() < profile["tail_share"]:
                fee_rate *= rng.uniform(profile["tail_multiplier"] * 0.5, profile["tail_multiplier"] * 1.5)
        fee_sats = max(0, int(round(fee_rate * vsize)))
        if fee_rate > 0 and fee_sats == 0:
            fee_sats = 1
        output_value_sats = rng.randint(20_000, 20_000_000)
        rows.append(_transaction_row(name, i, vsize, fee_sats, output_value_sats, rng))
    return rows


def _transaction_row(
    name: str,
    index: int,
    vsize_vb: int,
    fee_sats: int,
    output_value_sats: int,
    rng: random.Random,
) -> dict[str, Any]:
    txid = _fake_hash(f"tx:{name}:{index}")
    has_witness = rng.random() < 0.75
    return {
        "txid": txid,
        "wtxid": _fake_hash(f"wtx:{name}:{index}") if has_witness else txid,
        "position_in_block": index,
        "version": 2,
        "locktime": 0,
        "size_bytes": int(vsize_vb * (1.0 + rng.random() * 0.8)),
        "vsize_vb": vsize_vb,
        "weight_wu": vsize_vb * 4,
        "input_count": rng.randint(1, 6),
        "output_count": rng.randint(1, 4),
        "input_value_sats": output_value_sats + fee_sats,
        "output_value_sats": output_value_sats,
        "fee_sats": fee_sats,
        "fee_sat_vb": fee_sats / vsize_vb if vsize_vb else 0,
        "is_coinbase": False,
        "has_witness": has_witness,
    }


def _coinbase_row(name: str, profile: Mapping[str, Any], total_fees_sats: int) -> dict[str, Any]:
    subsidy_sats = _block_subsidy_sats(profile["height"])
    return {
        "txid": _fake_hash(f"coinbase:{name}"),
        "wtxid": _fake_hash(f"coinbase-w:{name}"),
        "position_in_block": 0,
        "version": 2,
        "locktime": 0,
        "size_bytes": 160,
        "vsize_vb": 135,
        "weight_wu": 540,
        "input_count": 1,
        "output_count": 1,
        "input_value_sats": None,
        "output_value_sats": subsidy_sats + total_fees_sats,
        "fee_sats": None,
        "fee_sat_vb": None,
        "is_coinbase": True,
        "has_witness": False,
    }


def _synthetic_vsize(profile: Mapping[str, Any], rng: random.Random) -> int:
    if rng.random() < profile["consolidation_share"]:
        return max(220, min(9500, int(rng.lognormvariate(math.log(2200), 0.7))))
    return max(110, min(1800, int(rng.lognormvariate(math.log(230), 0.55))))


def _block_subsidy_sats(height: int) -> int:
    halvings = height // 210_000
    if halvings >= 64:
        return 0
    return int(Decimal("50") * Decimal("100000000")) >> halvings


def _fake_hash(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def _rpc_auth(
    cookie: str | None,
    rpc_user: str | None,
    rpc_password: str | None,
) -> tuple[str, str] | None:
    if rpc_user and rpc_password:
        return rpc_user, rpc_password
    if not cookie:
        return None

    candidate = Path(cookie).expanduser()
    raw = candidate.read_text().strip() if candidate.exists() else cookie.strip()
    user, sep, password = raw.partition(":")
    if not sep:
        raise ValueError("RPC cookie must be a path or raw user:password string")
    return user, password


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()